The game provided is a slightly modified version of **Connect**. The board is of size *m × n*. On each turn, a player chooses a column, and one of their pieces is placed in that column. The winner is the first player who manages to align **five of their own pieces** consecutively—horizontally, vertically, or diagonally.

In this file there are two players, **Player1** and **Player2**, as well as several game-related functions. By reading these functions and examining the examples, you can understand exactly how the game works.

In the current code, both players are computer-controlled and make their moves at random. With a few small changes, you can modify the code so that you can play the game yourself. The steps of the assignment are as follows:

1. **Minimax player (65 points)**
   Write a function that uses the game’s scoring (evaluation) function so that a player can use the **minimax** algorithm and play a complete minimax game. The search depth for the algorithm should be **d = 4**.

2. **Alpha–beta pruning (15 points)**
   Add **alpha–beta pruning** to the function you wrote in part (1).

3. **Expectimax opponent (20 points)**
   Implement a function for the opposing player using **expectimax**, under the assumption that the opponent behaves randomly.

In [1]:
from termcolor import colored
import numpy as np
import random
import copy
import math
import sys

In [2]:
ROW_COUNT = 6
COLUMN_COUNT = 10

PLAYER1 = 0
PLAYER2 = 1

EMPTY = 0
PLAYER1_PIECE = 1
PLAYER2_PIECE = 2

WINDOW_LENGTH = 5

In [3]:
def create_board():
    board = np.zeros((ROW_COUNT, COLUMN_COUNT))
    return board


def print_board(board):
    flip_board = np.flip(board, 0)
    print()

    for r in range(ROW_COUNT):
        for c in range(COLUMN_COUNT):

            if flip_board[r][c] == PLAYER1_PIECE:
                print(colored(int(flip_board[r][c]), "grey", "on_red"), end=" ")

            elif flip_board[r][c] == PLAYER2_PIECE:
                print(colored(int(flip_board[r][c]), "grey", "on_blue"), end=" ")

            else:
                print(colored(int(flip_board[r][c]), "grey"), end=" ")

        print()


# Logic
def get_next_open_row(board, col):
    for r in range(ROW_COUNT):
        if board[r][col] == 0:
            return r


def drop_piece(board, row, col, piece):
    board = copy.copy(board)
    board[row][col] = piece
    return board


def is_valid_location(board, col):
    if col != None:
        return board[ROW_COUNT - 1][col] == 0
    return False


def get_valid_locations(board):
    valid_locations = []
    for col in range(COLUMN_COUNT):
        if is_valid_location(board, col):
            valid_locations.append(col)
    return valid_locations


def is_terminal_node(board):
    return (
        winning_move(board, PLAYER1_PIECE)
        or winning_move(board, PLAYER2_PIECE)
        or len(get_valid_locations(board)) == 0
    )


def winning_move(board, piece):
    # Check horizontal locations for win
    for c in range(COLUMN_COUNT - 4):
        for r in range(ROW_COUNT):
            if (
                board[r][c] == piece
                and board[r][c + 1] == piece
                and board[r][c + 2] == piece
                and board[r][c + 3] == piece
                and board[r][c + 4] == piece
            ):
                return True

    # Check vertical locations for win
    for c in range(COLUMN_COUNT):
        for r in range(ROW_COUNT - 4):
            if (
                board[r][c] == piece
                and board[r + 1][c] == piece
                and board[r + 2][c] == piece
                and board[r + 3][c] == piece
                and board[r + 4][c] == piece
            ):
                return True

    # Check positively sloped diaganols
    for c in range(COLUMN_COUNT - 4):
        for r in range(ROW_COUNT - 4):
            if (
                board[r][c] == piece
                and board[r + 1][c + 1] == piece
                and board[r + 2][c + 2] == piece
                and board[r + 3][c + 3] == piece
                and board[r + 4][c + 4] == piece
            ):
                return True

    # Check negatively sloped diaganols
    for c in range(COLUMN_COUNT - 4):
        for r in range(4, ROW_COUNT):
            if (
                board[r][c] == piece
                and board[r - 1][c + 1] == piece
                and board[r - 2][c + 2] == piece
                and board[r - 3][c + 3] == piece
                and board[r - 4][c + 4] == piece
            ):
                return True

    return False

In [4]:
def evaluate_window(window, piece):
    # This function attempts to evaluate a specific part of the board called window for a specific player.
    # So it counts the pieces of the player and also its oponent in that window and returns a score based on that.
    score = 0

    competitor_player_piece = PLAYER1_PIECE  # The competitor piece is 1 or 2?
    if piece == PLAYER2_PIECE:
        competitor_player_piece = PLAYER2_PIECE

    if window.count(piece) == 5:
        score += 100000
    elif window.count(piece) == 4 and window.count(EMPTY) == 1:
        score += 100
    elif window.count(piece) == 3 and window.count(EMPTY) == 2:
        score += 10
    elif window.count(piece) == 2 and window.count(EMPTY) == 3:
        score += 5

    if window.count(competitor_player_piece) == 4 and window.count(EMPTY) == 1:
        score -= 100
    elif window.count(competitor_player_piece) == 3 and window.count(EMPTY) == 2:
        score -= 10
    elif window.count(competitor_player_piece) == 2 and window.count(EMPTY) == 3:
        score -= 5
    return score


def score_position(board, piece):
    score = 0

    # Score horizontal position
    for r in range(ROW_COUNT):
        row_array = [int(i) for i in list(board[r, :])]
        for c in range(COLUMN_COUNT - 4):
            window = row_array[c : c + WINDOW_LENGTH]
            score += evaluate_window(window, piece)

    # Score vertical position
    for c in range(COLUMN_COUNT):
        col_array = [int(i) for i in list(board[:, c])]
        for r in range(ROW_COUNT - 4):
            window = col_array[r : r + WINDOW_LENGTH]
            score += evaluate_window(window, piece)

    # Score positively sloped diagonal position
    for r in range(ROW_COUNT - 4):
        for c in range(COLUMN_COUNT - 4):
            window = [board[r + i][c + i] for i in range(WINDOW_LENGTH)]
            score += evaluate_window(window, piece)

    # Score negatively sloped diagonal position
    for r in range(ROW_COUNT - 4):
        for c in range(COLUMN_COUNT - 4):
            window = [board[r - i][c + i] for i in range(WINDOW_LENGTH)]  # r + 4 - i
            score += evaluate_window(window, piece)

    return score

In [ ]:
def maximizer(board, depth):
    
    # This part returns the column and the score (utility) if the input board is the leaf of the tree
    if is_terminal_node(board):
        if winning_move(board, PLAYER1_PIECE):
            return 1000000
        elif winning_move(board, PLAYER2_PIECE):
            return -1000000
        else:
            return(0)
    
    # In this part the simulation ends based on the depth that was determined, but the node is not the leaf of the tree
    # so you require to use an evaluation function to estimate the score of that board
    elif depth == 0:
        return score_position(board, PLAYER1_PIECE)
    
    value = -math.inf
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them
    
    for col in cols: # for each valid column the simulation must be run to find its score
            
        # for the column you need to find the row in which the piece will be dropped
        row = get_next_open_row(board, col)
            
        # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
        new_board = drop_piece(board, row, col, PLAYER1_PIECE)
            
        score = minimizer(new_board, depth - 1)
        
        value = max(value, score) # the best value
    
    return value
    
    
    
def minimizer(board, depth):
    
    # This part returns the column and the score (utility) if the input board is the leaf of the tree
    if is_terminal_node(board):
        if winning_move(board, PLAYER1_PIECE):
            return 1000000
        elif winning_move(board, PLAYER2_PIECE):
            return -1000000
        else:
            return(0)
    
    # In this part the simulation ends based on the depth that was determined, but the node is not the leaf of the tree
    # so you require to use an evaluation function to estimate the score of that board
    elif depth == 0:
        return score_position(board, PLAYER2_PIECE)
    
    value = math.inf
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them
    
    for col in cols: # for each valid column the simulation must be run to find its score
            
        # for the column you need to find the row in which the piece will be dropped
        row = get_next_open_row(board, col)
        
        # In this line you drop the piece in the column to simulate the process
        new_board = drop_piece(board, row, col, PLAYER2_PIECE)
            
        # You need to build the tree and continue the simulation until the game is terminated or the depth gets to 0
        score = maximizer(new_board, depth - 1)
        
        value = min(value, score)
    
    return value 
    
    
            
    
def minimax_get_action(board, depth, player):
# This function uses the minimax algorithm and return the best column that the player can drop its piece in
    
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them

    
    if player == PLAYER1: # maximizer, player1 is considered as the maximum player
        value = -math.inf
        best_col = random.choice(cols) # a default value was considered as the best column, it will be changed
                                       # in the following lines
        scores = []
        for col in cols: # for each valid column the simulation must be run to find its score
            
            # for the column you need to find the row in which the piece will be dropped
            row = get_next_open_row(board, col)
            
            # In this line you drop the piece in the column to simulate the process
            new_board = drop_piece(board, row, col, PLAYER1_PIECE)
            # You need to build the tree and continue the simulation until the game is terminated or the depth gets to 0
            score = minimizer(new_board, depth - 1)
            scores.append(score)
            
        best_col = cols[scores.index(max(scores))] # the best column
        return best_col
    
    
    else: # minimizer, player2 is considered as the minimum player
        value = math.inf
        best_col = random.choice(cols) # a default value was considered as the best column, it will be changed
                                       # in the following lines
        scores = []
        for col in cols: # for each valid column the simulation must be run to find its score
            
            # for the column you need to find the row in which the piece will be dropped
            row = get_next_open_row(board, col)
            # In this line you drop the piece in the column to simulate the process
            new_board = drop_piece(board, row, col, PLAYER2_PIECE)
            # You need to build the tree and continue the simulation until the game is terminated or the depth gets to 0
            score = maximizer(new_board, depth - 1)
            scores.append(score) 
                                                  
            
        best_col = cols[scores.index(min(scores))] # the best column
        return best_col


In [ ]:
def maximizer_alpha_beta(board, depth, alpha, beta):
    
    # This part returns the column and the score (utility) if the input board is the leaf of the tree
    if is_terminal_node(board):
        if winning_move(board, PLAYER1_PIECE):
            return 1000000
        elif winning_move(board, PLAYER2_PIECE):
            return -1000000
        else:
            return(0)
    
    # In this part the simulation ends based on the depth that was determined, but the node is not the leaf of the tree
    # so you require to use an evaluation function to estimate the score of that board
    elif depth == 0:
        return score_position(board, PLAYER1_PIECE)
    
    value = -math.inf
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them
    
    for col in cols: # for each valid column the simulation must be run to find its score
            
        # for the column you need to find the row in which the piece will be dropped
        row = get_next_open_row(board, col)
        
        
        # In this line you drop the piece in the column to simulate the process
        new_board = drop_piece(board, row, col, PLAYER1_PIECE)
            
        # You need to build the tree and continue the simulation until the game is terminated or the depth gets to 0
        score = minimizer_alpha_beta(new_board, depth - 1, alpha, beta)
        
        value = max(value, score) # the best value
            
        if value > beta:
            return value
        alpha = max(alpha, value)
    
    return value
    
    
    
def minimizer_alpha_beta(board, depth, alpha, beta):
    
    # This part returns the column and the score (utility) if the input board is the leaf of the tree
    if is_terminal_node(board):
        if winning_move(board, PLAYER1_PIECE):
            return 1000000
        elif winning_move(board, PLAYER2_PIECE):
            return -1000000
        else:
            return(0)
    
    # In this part the simulation ends based on the depth that was determined, but the node is not the leaf of the tree
    # so you require to use an evaluation function to estimate the score of that board
    elif depth == 0:
        return score_position(board, PLAYER2_PIECE)
    
    value = math.inf
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them
    
    for col in cols: # for each valid column the simulation must be run to find its score
            
        # for the column you need to find the row in which the piece will be dropped
        row = get_next_open_row(board, col)
            
        # In this line you drop the piece in the column to simulate the process
        new_board = drop_piece(board, row, col, PLAYER2_PIECE)
        # You need to build the tree and continue the simulation until the game is terminated or the depth gets to 0
        score = maximizer_alpha_beta(new_board, depth - 1, alpha, beta)
        value = min(value, score) # the best value 
            
        if value < alpha:
            return value
        beta = min(beta, value)
    
    return value 
    
    
            
    
def minimax_alpha_beta_get_action(board, depth, player, alpha, beta):
# This function uses the minimax algorithm and return the best column that the player can drop its piece in
            
    alpha = -math.inf
    beta = math.inf
    
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them

    
    if player == PLAYER1: # maximizer, player1 is considered as the maximum player
        value = -math.inf
        best_col = random.choice(cols) # a default value was considered as the best column, it will be changed
                                       # in the following lines
        scores = []
        for col in cols: # for each valid column the simulation must be run to find its score
            
            # for the column you need to find the row in which the piece will be dropped
            row = get_next_open_row(board, col)
            #  In this line you drop the piece in the column to simulate the process.
            new_board = drop_piece(board, row, col, PLAYER1_PIECE)
            # You need to build the tree and continue the simulation until the game is terminated or the depth gets to 0
            score = minimizer_alpha_beta(new_board, depth - 1, alpha, beta)
            scores.append(score)
            value = max(value, score)
            alpha = max(alpha, value)
            
        best_col = cols[scores.index(max(scores))] # the best column
                
        return best_col
    
    
    else: # minimizer, player2 is considered as the minimum player
        value = math.inf
        best_col = random.choice(cols) # a default value was considered as the best column, it will be changed
                                       # in the following lines
        scores = []
        for col in cols: # for each valid column the simulation must be run to find its score
            
            row = get_next_open_row(board, col)
            new_board = drop_piece(board, row, col, PLAYER2_PIECE)
            score = maximizer_alpha_beta(new_board, depth - 1, alpha, beta)
            scores.append(score)
            value = min(value, score)
            beta = min(beta, value)
            
        best_col = cols[scores.index(min(scores))] # the best column
        return best_col


In [7]:
def exp_max(board, depth):
    """Max node for Player1"""
    if is_terminal_node(board):
        if winning_move(board, PLAYER1_PIECE):
            return 1000000
        elif winning_move(board, PLAYER2_PIECE):
            return -1000000
        else:
            return 0
    elif depth == 0:
        return score_position(board, PLAYER1_PIECE)

    value = -math.inf
    cols = get_valid_locations(board)

    for col in cols:
        row = get_next_open_row(board, col)
        new_board = drop_piece(board, row, col, PLAYER1_PIECE)
        score = exp_chance(new_board, depth - 1)
        value = max(value, score)

    return value


def exp_chance(board, depth):
    """Chance node for Player2 (random opponent)"""
    if is_terminal_node(board):
        if winning_move(board, PLAYER1_PIECE):
            return 1000000
        elif winning_move(board, PLAYER2_PIECE):
            return -1000000
        else:
            return 0
    elif depth == 0:
        return score_position(board, PLAYER1_PIECE)

    cols = get_valid_locations(board)
    if len(cols) == 0:
        return 0

    total_score = 0
    for col in cols:
        row = get_next_open_row(board, col)
        new_board = drop_piece(board, row, col, PLAYER2_PIECE)
        total_score += exp_max(new_board, depth - 1)

    return total_score / len(cols)  # expected value


def expectimax_get_action(board, depth, player):
    """Return the best column for expectimax AI (Player2)"""
    cols = get_valid_locations(board)

    if player == PLAYER2:
        scores = []
        for col in cols:
            row = get_next_open_row(board, col)
            new_board = drop_piece(board, row, col, PLAYER2_PIECE)
            score = exp_max(new_board, depth - 1)
            scores.append(score)

        best_score = max(scores)
        best_col = cols[scores.index(best_score)]
        return best_col
    else:
        return None

In [9]:
board = create_board()
print_board(board)
game_over = False
turn = random.randint(PLAYER1, PLAYER2)

while not game_over:

    if len(get_valid_locations(board)) == 0:
        label = 'TIE!'
        game_over = True
    
    
    if turn == PLAYER1 and not game_over:
        # col = random.choice(get_valid_locations(board))
        col = minimax_get_action(board, depth=4, player=PLAYER1)
        # col = minimax_alpha_beta_get_action(board, depth=4, player=PLAYER1, alpha=-math.inf, beta=math.inf)

        # UNCOMMENT THIS PART AND PLAY YOURSEF
        # col = int(input('Please enter the column you want to drop in (0 to 9): '))
        # while not col in get_valid_locations(board):
        #     col = int(input('Please enter the column you want to drop in (0 to 9): '))

        if is_valid_location(board, col):
            row = get_next_open_row(board, col)
            board = drop_piece(board, row, col, PLAYER1_PIECE)

            if winning_move(board, PLAYER1_PIECE):
                label = 'Player 1 wins!'
                game_over = True

            turn = 1 - turn
            print_board(board)
        
            
    if turn == PLAYER2 and not game_over:
        # col = random.choice(get_valid_locations(board))
        # col = minimax_get_action(board, depth=4, player=PLAYER2)
        # col = minimax_alpha_beta_get_action(board, depth=4, player=PLAYER2, alpha=-math.inf, beta=math.inf)
        col = expectimax_get_action(board, depth=3, player=PLAYER2)
        
        # UNCOMMENT THIS PART AND PLAY YOURSEF
        # col = int(input('Please enter the column you want to drop in (0 to 9): '))
        # while not col in get_valid_locations(board):
            # col = int(input('Please enter the column you want to drop in (0 to 9): '))

        if is_valid_location(board, col):
            row = get_next_open_row(board, col)
            board = drop_piece(board, row, col, PLAYER2_PIECE)

            if winning_move(board, PLAYER2_PIECE):
                label = 'Player 2 wins!'
                game_over = True

            turn = 1 - turn
            print_board(board)
            
                   
    if game_over:
        label = label + ' Game over'
        print ('\n', colored(label, 'green'))


0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 

0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 

0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 

0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 

0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 

0 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 

1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 

1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 2 0